## Setup and Configuration

In [1]:
"""
RAFM Irradiation Analysis - Setup Cell

This notebook handles ALARA model creation and execution for RAFM steel
irradiation experiments.

Directory structure (rafm_irradiation_ldrd):
├── alara_inputs/              # ALARA input data files
│   ├── mcnp_inputs/           # MCNP input files
│   └── runtpe.h5              # MCNP mesh tally data
├── irradiation_QG_processed/  # Processed experimental data
│   ├── RAFM1/, RAFM3/, RAFM4/ # Gamma spectra by sample set
│   └── flux_wires/            # Flux wire measurement data
├── raw_gamma_spec/            # Raw gamma spectra files (.ASC)
├── scripts/                   # Python support modules
├── alara_output/              # Generated output directory
└── RAFM_ALARA_Modeling.ipynb  # This notebook
"""

import os
import sys
import subprocess
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============== Directory Structure ==============
PROJECT_ROOT = Path.cwd()
ALARA_ROOT = PROJECT_ROOT.parent

# Input directories
INPUTS_DIR = PROJECT_ROOT / "alara_inputs"
MCNP_INPUTS_DIR = INPUTS_DIR / "mcnp_inputs"

# Existing data directories
EXPERIMENTAL_DIR = PROJECT_ROOT / "irradiation_QG_processed"
FLUX_WIRES_DIR = EXPERIMENTAL_DIR / "flux_wires"
SPECTRA_DIR = PROJECT_ROOT / "raw_gamma_spec"

# Scripts directory
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
TOOLS_DIR = ALARA_ROOT / "tools"

# Output directories
OUTPUT_DIR = PROJECT_ROOT / "alara_output"
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'decay_curves').mkdir(exist_ok=True)
(OUTPUT_DIR / 'activity_plots').mkdir(exist_ok=True)
(OUTPUT_DIR / 'snr_optimization').mkdir(exist_ok=True)

# ============== Python Path Setup ==============
for path in [SCRIPTS_DIR, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# ============== Import ALARA Tools ==============
from alara_output_processing import FileParser, ALARADFrame, DataLibrary, SECONDS_CONV, convert_times
print(f"✓ ALARA output processing tools loaded from: {TOOLS_DIR}")

from alara_output_processing import alara_output_plotting as aop_plotting
print(f"✓ ALARA plotting tools loaded")

# ============== Import Our Support Modules ==============
from nuclear_data import get_half_life, get_gamma_info, using_ensdf, get_data_source
from element_data import ELEMENT_Z, Z_TO_ELEMENT, element_to_z, z_to_element
from isotope_utils import canonical_iso, format_iso_pretty, get_half_life_days, has_gamma_emission

from plotting import (
    MATERIAL_COLORS, COOLING_COLORS, SOURCE_COLORS, LABELS,
    apply_standard_style,
)
apply_standard_style()
print(f"✓ Consolidated plotting module loaded")

print(f"✓ Nuclear data loaded: {get_data_source()}")
print(f"✓ Element data loaded: {len(ELEMENT_Z)} elements from elelib")

# ============== Import MCNP Workflow Modules ==============
MCNP_WORKFLOW_DIR = ALARA_ROOT / "MCNP_ALARA_Workflow"
if str(MCNP_WORKFLOW_DIR) not in sys.path:
    sys.path.append(str(MCNP_WORKFLOW_DIR))

import alara_comparison
print(f"✓ ALARA comparison modules loaded")

# ============== Configuration ==============
MCNP_INPUT = MCNP_INPUTS_DIR / 'whale_J_core_clean_loc.i'
H5_FILE = INPUTS_DIR / 'runtpe.h5'
SPECTRUM_FILE = SPECTRA_DIR / 'flux_wires' / 'spectrum_vit_j.csv'

print(f"\n{'='*50}")
print(f"RAFM ALARA Modeling - Configuration")
print(f"{'='*50}")
print(f"Project root:      {PROJECT_ROOT}")
print(f"MCNP input:        {MCNP_INPUT}")
print(f"HDF5 file:         {H5_FILE}")
print(f"Spectrum file:     {SPECTRUM_FILE}")
print(f"Experimental data: {EXPERIMENTAL_DIR}")
print(f"Output dir:        {OUTPUT_DIR}")

## Step 0: Load Experimental Data

Import measured activation data and compute cooling times for ALARA simulations.
This data is loaded early to allow cooling time determination for subsequent steps.

In [2]:
# Ensure output directory exists
if 'alara_output_dir' not in globals():
    if 'OUTPUT_DIR' in globals():
        alara_output_dir = Path(OUTPUT_DIR)
    else:
        alara_output_dir = Path('alara_output')

alara_output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {alara_output_dir.absolute()}")

# ==============================================================================
# IMPORT UTILITIES FROM STANDALONE SCRIPTS
# ==============================================================================
import importlib
import experimental_data_loader
importlib.reload(experimental_data_loader)

from isotope_utils import (
    canonical_iso, format_iso_pretty,
    get_half_life_days, get_half_life_seconds,
    half_life_to_lambda, parse_activity_unit,
    LN2, AVOGADRO,
)
print("✓ Loaded isotope_utils")

from flux_wire_loader import (
    load_flux_wires, get_metadata_df, WIRE_METADATA
)
print("✓ Loaded flux_wire_loader")

from experimental_data_loader import (
    load_experimental_data, parse_experimental_file,
    experimental_to_dataframe, get_top_isotopes,
    SAMPLE_TO_MATERIAL, MATERIAL_TO_TALLY, COOLING_TIME_MAP
)
print("✓ Loaded experimental_data_loader")
print(f"  SAMPLE_TO_MATERIAL: {SAMPLE_TO_MATERIAL}")

# ==============================================================================
# LOAD FLUX WIRE DATA
# ==============================================================================
print("\n" + "=" * 80)
print("FLUX WIRE DATA")
print("=" * 80)

flux_wires_dir = FLUX_WIRES_DIR if 'FLUX_WIRES_DIR' in dir() else Path('irradiation_QG_processed/flux_wires')

if flux_wires_dir.exists():
    flux_wires_df = load_flux_wires(
        flux_wires_dir=str(flux_wires_dir),
        sample_filter=r'-1($|_)',
        default_irradiation_s=7200,
        output_csv=str(alara_output_dir / 'flux_wires_activity.csv')
    )
    meta_df = get_metadata_df()
    
    if not flux_wires_df.empty:
        display(flux_wires_df.head())
    else:
        print("No flux wire data loaded.")
else:
    print(f"Flux wires directory not found: {flux_wires_dir}")
    flux_wires_df = pd.DataFrame()

# ==============================================================================
# LOAD EXPERIMENTAL DATA FROM BOTH RAFM3 AND RAFM4
# ==============================================================================
print("\n" + "=" * 80)
print("EXPERIMENTAL DATA")
print("=" * 80)

experimental_data = {}

# Load RAFM3 data (measurements after first 3s irradiation)
exp_data_dir_3 = EXPERIMENTAL_DIR / 'RAFM3' if 'EXPERIMENTAL_DIR' in dir() else Path('irradiation_QG_processed/RAFM3')

exp_data_3 = {}
if exp_data_dir_3.exists():
    print(f"\n--- Loading RAFM3 (3s irradiation) from {exp_data_dir_3} ---")
    exp_data_3 = load_experimental_data(
        exp_data_dir=str(exp_data_dir_3),
        verbose=True
    )
    for mat, times in exp_data_3.items():
        if mat not in experimental_data:
            experimental_data[mat] = {}
        for time_key, isotopes in times.items():
            experimental_data[mat][time_key] = isotopes
    print(f"\n✓ RAFM3: Loaded data for {len(exp_data_3)} materials")
else:
    print(f"RAFM3 directory not found: {exp_data_dir_3}")

# Load RAFM4 data (measurements after 2hr irradiation)
exp_data_dir_4 = EXPERIMENTAL_DIR / 'RAFM4' if 'EXPERIMENTAL_DIR' in dir() else Path('irradiation_QG_processed/RAFM4')

exp_data_4 = {}
if exp_data_dir_4.exists():
    print(f"\n--- Loading RAFM4 (2hr irradiation) from {exp_data_dir_4} ---")
    exp_data_4 = load_experimental_data(
        exp_data_dir=str(exp_data_dir_4),
        verbose=True
    )
    for mat, times in exp_data_4.items():
        if mat not in experimental_data:
            experimental_data[mat] = {}
        for time_key, isotopes in times.items():
            experimental_data[mat][time_key] = isotopes
    print(f"\n✓ RAFM4: Loaded data for {len(exp_data_4)} materials")
else:
    print(f"RAFM4 directory not found: {exp_data_dir_4}")

# Create combined DataFrame
if experimental_data:
    exp_df = experimental_to_dataframe(experimental_data)
    print(f"\n✓ Combined experimental data: {len(exp_df)} total measurements")
    
    print("\nData summary by material and cooling time:")
    for mat in sorted(experimental_data.keys()):
        times = sorted(experimental_data[mat].keys())
        isotope_counts = {t: len(experimental_data[mat][t]) for t in times}
        print(f"  {mat}: {isotope_counts}")
else:
    exp_df = pd.DataFrame()
    print("No experimental data loaded.")

print("\n" + "=" * 80)
print("DATA LOADING COMPLETE")
print("=" * 80)

In [3]:
# Global defaults: ensure correct default irradiation time (2 hours)
DEFAULT_IRRADIATION_SECONDS = 7200  # 2 hours
DEFAULT_DECAY_SECONDS = 0
print(f"DEFAULT_IRRADIATION_SECONDS set to {DEFAULT_IRRADIATION_SECONDS} seconds")

## Step 0.5: Calculate Sample-Specific Irradiation Schedules

Parse the experimental gamma spectroscopy files to extract measurement timestamps,
then calculate the exact cooling times for each sample.

**Irradiation Timeline:**
- **First irradiation (3s):** Occurs 6 minutes before the first 300s measurement for each sample
- **RAFM3 measurements:** 300s, 2h, 24h, 4d after the 3s irradiation (July 31, 2025)
- **Second irradiation (2h):** August 4, 2025 at 1:00 PM for ALL samples
- **RAFM4 measurements:** 15 days after the 2h irradiation (August 19-20, 2025)

In [4]:
# ==============================================================================
# PARSE EXPERIMENTAL GAMMA SPEC FILES TO GET MEASUREMENT TIMESTAMPS
# ==============================================================================
import re
from datetime import datetime, timedelta

# Import the schedule calculation utilities from our scripts
from sample_irradiation_schedule import (
    parse_date_from_gamma_file, 
    format_seconds_to_alara,
    SAMPLE_LETTERS, SAMPLE_TO_MATERIAL_MAP, MATERIAL_TO_TALLY_MAP,
    RAFM3_PATTERNS, RAFM4_PATTERNS,
    FIRST_IRR_DURATION, FIRST_IRR_OFFSET_BEFORE_300S,
    SECOND_IRR_DURATION, SECOND_IRR_START, SECOND_IRR_END,
)

print("✓ Loaded schedule utilities from sample_irradiation_schedule.py")

# ==============================================================================
# PARSE ALL MEASUREMENT TIMES FROM EXPERIMENTAL FILES
# ==============================================================================
print("\n" + "=" * 80)
print("PARSING MEASUREMENT TIMESTAMPS FROM EXPERIMENTAL FILES")
print("=" * 80)

# Print sample mapping for reference
print(f"\nSample mapping:")
for sample, material in SAMPLE_TO_MATERIAL_MAP.items():
    print(f"  Sample {sample} → {material} → {MATERIAL_TO_TALLY_MAP[material]}")

# Parse RAFM3 and RAFM4 files
rafm3_dir = EXPERIMENTAL_DIR / 'RAFM3'
rafm4_dir = EXPERIMENTAL_DIR / 'RAFM4'

measurement_times = {}
for sample in SAMPLE_LETTERS:
    measurement_times[sample] = {}
    
    # RAFM3 files
    for pattern, label in RAFM3_PATTERNS.items():
        filepath = rafm3_dir / f"RAFM3-{sample}_{pattern}.txt"
        if filepath.exists():
            meas_time = parse_date_from_gamma_file(filepath)
            if meas_time:
                measurement_times[sample][label] = meas_time
                
    # RAFM4 files
    for pattern, label in RAFM4_PATTERNS.items():
        filepath = rafm4_dir / f"RAFM4-{sample}_{pattern}.txt"
        if filepath.exists():
            meas_time = parse_date_from_gamma_file(filepath)
            if meas_time:
                measurement_times[sample][label] = meas_time

# Print parsed times
for sample in SAMPLE_LETTERS:
    print(f"\nSample {sample} ({SAMPLE_TO_MATERIAL_MAP[sample]}):")
    for label, meas_time in sorted(measurement_times[sample].items()):
        print(f"  {label:6s}: {meas_time}")

# ==============================================================================
# CALCULATE IRRADIATION SCHEDULES FOR EACH SAMPLE
# ==============================================================================
print("\n" + "=" * 80)
print("CALCULATING SAMPLE-SPECIFIC IRRADIATION SCHEDULES")
print("=" * 80)

print(f"\nSecond irradiation (common to all samples):")
print(f"  Start: {SECOND_IRR_START}")
print(f"  End:   {SECOND_IRR_END}")
print(f"  Duration: 2 hours")

sample_schedules = {}

for sample in SAMPLE_LETTERS:
    material = SAMPLE_TO_MATERIAL_MAP[sample]
    tally = MATERIAL_TO_TALLY_MAP[material]
    
    schedule = {
        'sample': sample,
        'material': material,
        'tally': tally,
        'phase1': None,
        'phase2': None,
    }
    
    # Phase 1: 3-second irradiation (before RAFM3 300s measurement)
    irr1_end = None
    if '300s' in measurement_times[sample]:
        meas_300s = measurement_times[sample]['300s']
        
        # First irradiation ENDS 5 minutes before the 300s measurement
        irr1_end = meas_300s - timedelta(seconds=FIRST_IRR_OFFSET_BEFORE_300S)
        irr1_start = irr1_end - timedelta(seconds=FIRST_IRR_DURATION)
        
        phase1_cooling = []
        for label in ['300s', '2h', '24h', '4d']:
            if label in measurement_times[sample]:
                meas_time = measurement_times[sample][label]
                cooling_seconds = (meas_time - irr1_end).total_seconds()
                phase1_cooling.append({
                    'label': label,
                    'seconds': cooling_seconds,
                    'alara_format': format_seconds_to_alara(cooling_seconds),
                    'measurement_time': meas_time,
                })
        
        schedule['phase1'] = {
            'irradiation_time': "3 s",
            'irradiation_seconds': FIRST_IRR_DURATION,
            'irradiation_start': irr1_start,
            'irradiation_end': irr1_end,
            'cooling_times': phase1_cooling,
        }
    
    # Phase 2: 2-hour irradiation (before RAFM4 15d measurement)
    if '15d' in measurement_times[sample]:
        meas_15d = measurement_times[sample]['15d']
        cooling_15d_seconds = (meas_15d - SECOND_IRR_END).total_seconds()
        
        # Calculate delay from Phase 1 end to Phase 2 START (not cooling time!)
        # This is the key fix: Phase 2 starts at a fixed time (Aug 4, 1PM)
        # The delay is from when Phase 1 ended to when Phase 2 starts
        delay_from_phase1_seconds = None
        delay_from_phase1_alara = None
        if irr1_end is not None:
            delay_from_phase1_seconds = (SECOND_IRR_START - irr1_end).total_seconds()
            delay_from_phase1_alara = format_seconds_to_alara(delay_from_phase1_seconds)
        
        schedule['phase2'] = {
            'irradiation_time': "2 h",
            'irradiation_seconds': SECOND_IRR_DURATION,
            'irradiation_start': SECOND_IRR_START,
            'irradiation_end': SECOND_IRR_END,
            'delay_from_phase1_seconds': delay_from_phase1_seconds,
            'delay_from_phase1_alara': delay_from_phase1_alara,
            'cooling_times': [{
                'label': '15d',
                'seconds': cooling_15d_seconds,
                'alara_format': format_seconds_to_alara(cooling_15d_seconds),
                'measurement_time': meas_15d,
            }],
        }
    
    sample_schedules[sample] = schedule

# Print calculated schedules
for sample, schedule in sample_schedules.items():
    print(f"\n{'='*60}")
    print(f"SAMPLE {sample}: {schedule['material']} ({schedule['tally']})")
    print(f"{'='*60}")
    
    if schedule['phase1']:
        p1 = schedule['phase1']
        print(f"\n  PHASE 1: {p1['irradiation_time']} irradiation")
        print(f"    Irradiation: {p1['irradiation_start']} → {p1['irradiation_end']}")
        print(f"    Cooling times (from EOI-1):")
        for ct in p1['cooling_times']:
            print(f"      {ct['label']:6s} → {ct['alara_format']:>14s} ({ct['seconds']:.0f}s)")
    
    if schedule['phase2']:
        p2 = schedule['phase2']
        print(f"\n  PHASE 2: {p2['irradiation_time']} irradiation")
        print(f"    Irradiation: {p2['irradiation_start']} → {p2['irradiation_end']}")
        if p2.get('delay_from_phase1_alara'):
            print(f"    Delay from Phase 1 end: {p2['delay_from_phase1_alara']} ({p2['delay_from_phase1_seconds']:.0f}s)")
        print(f"    Cooling times (from EOI-2):")
        for ct in p2['cooling_times']:
            print(f"      {ct['label']:6s} → {ct['alara_format']:>14s} ({ct['seconds']:.0f}s)")

print("\n✓ Sample-specific schedules calculated from experimental file timestamps")

## Step 0.6: Extract Cylindrical Geometry from H5 File

Read the mesh tally geometry from the HDF5 file to determine the sample dimensions.
ALARA geometry will be generated as a cylinder matching these dimensions.

In [ ]:
# ==============================================================================
# EXTRACT CYLINDRICAL GEOMETRY FROM H5 MESH TALLY
# ==============================================================================
# Use the geometry extraction function from mcnp_to_alara.py

print("=" * 80)
print("EXTRACTING GEOMETRY FROM HDF5 MESH TALLY")
print("=" * 80)

# Import the geometry extraction function from mcnp_to_alara
from mcnp_to_alara import get_all_mesh_geometries

h5_path = H5_FILE if 'H5_FILE' in dir() else Path('alara_inputs/runtpe.h5')
print(f"\nHDF5 file: {h5_path}\n")

# Get all mesh geometries
mesh_geometry = get_all_mesh_geometries(str(h5_path), verbose=True)

# Map to sample geometry using schedule info
print("\n" + "=" * 80)
print("GEOMETRY SUMMARY FOR ALARA")
print("=" * 80)

SAMPLE_GEOMETRY = {}
for sample, schedule in sample_schedules.items():
    tally_str = schedule['tally']
    tally_num = int(tally_str.replace('tally_', ''))
    
    if tally_num in mesh_geometry:
        geom = mesh_geometry[tally_num]
        SAMPLE_GEOMETRY[sample] = {
            'tally': tally_str,
            'material': schedule['material'],
            'mesh_type': geom.get('mesh_type', 'unknown'),
            'radius': geom.get('radius', 0.5),
            'height': geom.get('height', 1.0),
            'volume_cm3': geom.get('volume_cm3', 1.0),
            'n_voxels': geom.get('n_voxels', 64),
        }
        sg = SAMPLE_GEOMETRY[sample]
        print(f"\nSample {sample} ({sg['material']}):")
        print(f"  Mesh type: {sg['mesh_type']}")
        print(f"  Radius: {sg['radius']:.4f} cm")
        print(f"  Height: {sg['height']:.4f} cm")
        print(f"  Volume: {sg['volume_cm3']:.6f} cm³")
        print(f"  Voxels: {sg['n_voxels']}")

print("\n✓ Geometry extracted from HDF5 mesh tallies using mcnp_to_alara.get_all_mesh_geometries()")

## Step 1: Generate ALARA Input Files with Sample-Specific Schedules

This step generates ALARA input files for each sample using:
- **Cylindrical geometry** matching the mesh tally dimensions from HDF5
- **Sample-specific cooling times** calculated from experimental gamma spec timestamps
- **Dual-phase irradiation**: 3s (RAFM3) + 2h (RAFM4) schedules
- Custom element library with MCNP isotopic ratios

In [6]:
# ==============================================================================
# GENERATE ALARA INPUT FILES WITH SAMPLE-SPECIFIC SCHEDULES AND GEOMETRY
# ==============================================================================
import json

print("=" * 80)
print("GENERATING ALARA INPUT FILES")
print("=" * 80)

# Create a schedule file that the mcnp_to_alara.py script can read
schedules_file = OUTPUT_DIR / 'sample_schedules.json'

# Convert datetime objects to strings for JSON serialization
schedules_for_json = {}
for sample, schedule in sample_schedules.items():
    s = {
        'sample': schedule['sample'],
        'material': schedule['material'],
        'tally': schedule['tally'],
    }
    
    if schedule['phase1']:
        p1 = schedule['phase1']
        s['phase1'] = {
            'irradiation_time': p1['irradiation_time'],
            'irradiation_seconds': p1['irradiation_seconds'],
            'irradiation_end_str': str(p1['irradiation_end']),  # For delay calculation
            'cooling_times': [
                {'label': ct['label'], 'seconds': ct['seconds'], 'alara_format': ct['alara_format']}
                for ct in p1['cooling_times']
            ]
        }
    
    if schedule['phase2']:
        p2 = schedule['phase2']
        s['phase2'] = {
            'irradiation_time': p2['irradiation_time'],
            'irradiation_seconds': p2['irradiation_seconds'],
            'cooling_times': [
                {'label': ct['label'], 'seconds': ct['seconds'], 'alara_format': ct['alara_format']}
                for ct in p2['cooling_times']
            ],
        }
        # Add the delay from Phase 1 end to Phase 2 start (THIS IS THE KEY FIX!)
        # This is NOT the same as the last Phase 1 cooling time!
        if p2.get('delay_from_phase1_seconds') is not None:
            s['phase2']['delay_from_phase1_seconds'] = p2['delay_from_phase1_seconds']
            s['phase2']['delay_from_phase1_alara'] = p2['delay_from_phase1_alara']
    
    schedules_for_json[sample] = s

# Add geometry information
geometry_for_json = {}
if 'SAMPLE_GEOMETRY' in dir():
    for sample, geom in SAMPLE_GEOMETRY.items():
        geometry_for_json[sample] = {
            'tally': geom['tally'],
            'material': geom['material'],
            'mesh_type': geom['mesh_type'],
            'radius_cm': geom['radius'],
            'height_cm': geom['height'],
            'volume_cm3': geom['volume_cm3'],
        }

# Save combined config
config = {
    'schedules': schedules_for_json,
    'geometry': geometry_for_json,
    'irradiation': {
        'phase1_duration_s': FIRST_IRR_DURATION,
        'phase1_offset_before_300s_s': FIRST_IRR_OFFSET_BEFORE_300S,
        'phase2_duration_s': SECOND_IRR_DURATION,
        'phase2_start': str(SECOND_IRR_START),
        'phase2_end': str(SECOND_IRR_END),
    }
}

with open(schedules_file, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✓ Saved sample configuration to: {schedules_file}")

# Print the Phase 2 delays to verify they are correct
print("\n  Phase 2 delays from Phase 1 end (should be ~4.13-4.15 days, NOT 4.0 days):")
for sample, s in schedules_for_json.items():
    if 'phase2' in s and s['phase2'].get('delay_from_phase1_alara'):
        print(f"    {sample} ({s['material']}): {s['phase2']['delay_from_phase1_alara']}")

# Run the main workflow script with sample-specific cooling times
exp_data_dir = str(EXPERIMENTAL_DIR / 'RAFM3')
spectrum_csv = str(SPECTRUM_FILE)
mcnp_input = str(MCNP_INPUT)
h5_file = str(H5_FILE)
output_dir = str(OUTPUT_DIR)

mcnp_to_alara_script = str(SCRIPTS_DIR / 'mcnp_to_alara.py')

cmd = [
    sys.executable, mcnp_to_alara_script,
    '--mcnp', mcnp_input,
    '--spectrum-csv', spectrum_csv,
    '--h5', h5_file,
    '--output', output_dir,
    '--run-alara',
    '--library', 'fendl2',
    '--exp-data-dir', exp_data_dir,
    '--geometry', 'cylindrical',  # Use cylindrical geometry
]

print(f"\nCommand: {' '.join(cmd)}\n")
print("=" * 80)
print("RUNNING MCNP TO ALARA WORKFLOW")
print("=" * 80)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
process.wait()

print(f"\nReturn code: {process.returncode}")

## Step 2: Analyze Flux Distribution Across Voxels

Compare the neutron flux in different voxels to understand spatial variation.

In [7]:
# ==============================================================================
# ANALYZE FLUX SPECTRA FROM MESH TALLIES
# ==============================================================================
import h5py
import importlib

material_names = {
    'tally_85214': 'CNA',
    'tally_85224': 'EUROFER97_4',
    'tally_85234': 'EUROFER97_3',
    'tally_85244': 'EUROFER97_2'
}

try:
    import flux_mesh_analysis
    importlib.reload(flux_mesh_analysis)
    from flux_mesh_analysis import (
        analyze_flux_spectrum, analyze_all_mesh_tallies,
        compare_materials_flux, print_flux_summary, plot_flux_spectra
    )
    HAS_FLUX_MESH = True
    print("✓ Loaded flux_mesh_analysis module")
except ImportError as e:
    print(f"⚠ Could not import flux_mesh_analysis: {e}")
    HAS_FLUX_MESH = False
    
    def analyze_flux_spectrum(h5_file, tally_num):
        with h5py.File(h5_file, 'r') as f:
            mt = f['results']['mesh_tally'][f'mesh_tally_{tally_num}']
            raw_flux = mt['mean'][:]
            flux_groups = raw_flux[:-1]
            total_flux_data = raw_flux[-1]
            n_groups = flux_groups.shape[0]
            flux = flux_groups.reshape(n_groups, -1)
            n_voxels = flux.shape[1]
            total_flux = total_flux_data.flatten()
            mean_spectrum = flux.mean(axis=1)
            try:
                energy_bins = mt['grid_energy'][:]
            except KeyError:
                energy_bins = np.arange(n_groups + 1)
            return {
                'n_groups': n_groups, 'n_voxels': n_voxels,
                'mean_spectrum': mean_spectrum, 'energy_bins': energy_bins,
                'total_flux_per_voxel': total_flux, 'flux_data': flux
            }

# Analyze each tally
tallies = [85214, 85224, 85234, 85244]
flux_analysis = {}

print("=" * 80)
print("ANALYZING FLUX SPECTRA FROM MESH TALLIES")
print("=" * 80)

for tally_num in tallies:
    flux_analysis[tally_num] = analyze_flux_spectrum(H5_FILE, tally_num)
    fa = flux_analysis[tally_num]
    print(f"✓ Tally {tally_num} ({material_names.get(f'tally_{tally_num}', 'unknown')}):")
    print(f"    Energy groups: {fa['n_groups']}")
    print(f"    Voxels: {fa['n_voxels']}")
    print(f"    Mean total flux: {fa['mean_spectrum'].sum():.3e} n/cm²/s")

# Plot energy spectrum comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (tally_num, fa) in enumerate(flux_analysis.items()):
    if idx >= 4:
        break
    ax = axes[idx]
    
    if len(fa['energy_bins']) > 1:
        e_centers = (fa['energy_bins'][:-1] + fa['energy_bins'][1:]) / 2
    else:
        e_centers = np.arange(fa['n_groups'])
    
    valid = fa['mean_spectrum'] > 0
    ax.loglog(e_centers[valid], fa['mean_spectrum'][valid], 'b-', lw=1)
    ax.set_xlabel('Energy (MeV)')
    ax.set_ylabel('Flux (n/cm²/s)')
    ax.set_title(f'Tally {tally_num} ({material_names.get(f"tally_{tally_num}", "unknown")})')
    ax.grid(True, alpha=0.3)

plt.suptitle('Energy Spectra Comparison Across Materials', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'flux_spectra_comparison.png', dpi=150, bbox_inches='tight')
print(f'\nFigure saved to: {OUTPUT_DIR / "flux_spectra_comparison.png"}')
plt.show()

if HAS_FLUX_MESH:
    print_flux_summary(flux_analysis, material_names)

In [8]:
# Compare total flux between materials
print("\n" + "="*80)
print("FLUX COMPARISON BETWEEN MATERIALS")
print("="*80)

flux_comparison = []
for tally_num, flux_data in flux_analysis.items():
    material = material_names.get(f'tally_{tally_num}', 'unknown')
    total_flux_per_voxel = flux_data['total_flux_per_voxel']
    mean_total = total_flux_per_voxel.mean()
    std_total = total_flux_per_voxel.std()
    
    flux_comparison.append({
        'Tally': tally_num,
        'Material': material,
        'Mean Flux (n/cm²/src)': mean_total,
        'Min Flux': total_flux_per_voxel.min(),
        'Max Flux': total_flux_per_voxel.max(),
        'Spatial Std Dev': std_total,
        'Spatial CV (%)': std_total/mean_total*100 if mean_total > 0 else 0,
    })

flux_df = pd.DataFrame(flux_comparison)
print("\n" + flux_df.to_string(index=False))

# Bar chart of mean flux
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(flux_df['Material'], flux_df['Mean Flux (n/cm²/src)'])
ax.set_ylabel('Mean Flux (n/cm²/src)')
ax.set_xlabel('Material')
ax.set_title('Comparison of Mean Neutron Flux by Material')
ax.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
ax.errorbar(range(len(flux_df)), flux_df['Mean Flux (n/cm²/src)'], 
            yerr=flux_df['Spatial Std Dev'], fmt='none', color='red', capsize=5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'flux_comparison_bar.png', dpi=150, bbox_inches='tight')
print(f'\nFigure saved to: {OUTPUT_DIR / "flux_comparison_bar.png"}')
plt.show()

## Step 3: Post-Process ALARA Output

Run the post-processing script to generate CSV files with voxel-averaged results and uncertainties.

In [9]:
# Run post-processing on all tally directories
print("Running post-processing on all tallies...")

postprocess_script = str(SCRIPTS_DIR / 'alara_multizone_postprocess.py') if 'SCRIPTS_DIR' in dir() else 'scripts/alara_multizone_postprocess.py'
output_dir = str(OUTPUT_DIR) if 'OUTPUT_DIR' in dir() else 'alara_output'

cmd = [
    sys.executable, postprocess_script,
    '--input', output_dir,
    '--output', output_dir
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## Step 4: Load and Explore Results

Load the CSV files and explore the activation results.

In [10]:
# Load all activity CSV files
activity_data = {}

material_names = {
    'tally_85214': 'CNA',
    'tally_85224': 'EUROFER97_C',
    'tally_85234': 'EUROFER97_B',
    'tally_85244': 'EUROFER97_A'
}

for csv_file in sorted(glob.glob(os.path.join(OUTPUT_DIR, 'tally_*_Bq_per_cm3.csv'))):
    tally_name = os.path.basename(csv_file).replace('_Bq_per_cm3.csv', '')
    if tally_name == 'tally_85114':
        continue
    material = material_names.get(tally_name, 'unknown')
    df = pd.read_csv(csv_file)
    activity_data[tally_name] = {'material': material, 'data': df}
    print(f"\n{tally_name} ({material}): {len(df)} isotopes")
    print(df.head(5))

print(f"\nLoaded {len(activity_data)} activity datasets")

## Step 5: Identify Key Activation Products

Find the most significant activation products at experimental measurement times (300s, 2h, 24h, 4d, 15d).

In [ ]:
# ==============================================================================
# IDENTIFY KEY ACTIVATION PRODUCTS
# ==============================================================================
# Use alara_output_processing for proper filtering and analysis

VALID_MEASUREMENT_TIMES = ['300s', '2h', '24h', '4d', '15d']
VALID_MEASUREMENT_SECONDS = [300, 7200, 86400, 345600, 1296000]

def get_top_isotopes_from_csv(df, n=10, time_key='300s'):
    """
    Get top N isotopes by activity at a specified measurement time from CSV data.
    This works with post-processed CSV files from alara_multizone_postprocess.py.
    
    For ALARADFrame data, use aop_plotting.preprocess_data() instead.
    """
    mean_cols = [c for c in df.columns if c.startswith('mean_')]
    if not mean_cols:
        return pd.DataFrame()
    
    # Find the column matching the time key
    target_col = None
    for col in mean_cols:
        if 'shutdown' in col.lower():
            continue
        if time_key.lower().replace(' ', '') in col.lower().replace(' ', ''):
            target_col = col
            break
    
    if target_col is None:
        target_col = next((c for c in mean_cols if 'shutdown' not in c.lower()), mean_cols[0])
    
    return df.nlargest(n, target_col)[['isotope', target_col]].rename(
        columns={'isotope': 'Isotope', target_col: f'Activity (Bq/cm³) at {time_key}'})

# Show top isotopes for each tally
for tally_name, data in activity_data.items():
    material = data['material']
    df = data['data']
    
    print(f"\n{'='*60}")
    print(f"TOP ACTIVATION PRODUCTS: {tally_name} ({material})")
    print(f"At first measurement time (300s after irradiation)")
    print(f"{'='*60}")
    
    top = get_top_isotopes_from_csv(df, 10, time_key='300s')
    if not top.empty:
        print(top.to_string(index=False))

## Step 5b: Uncertainty Analysis

Analyze the uncertainties in the voxel-averaged activation results.

In [12]:
print("="*80)
print("UNCERTAINTY ANALYSIS")
print("="*80)
print("\nThe uncertainties represent the standard error of the mean (SEM)")
print("across all voxels in each mesh tally. This captures spatial variation.")

for tally_name, data in activity_data.items():
    material = data['material']
    df = data['data']
    
    print(f"\n{'-'*60}")
    print(f"{material} ({tally_name})")
    print(f"{'-'*60}")
    
    mean_cols = [c for c in df.columns if c.startswith('mean_')]
    sem_cols = [c for c in df.columns if c.startswith('sem_')]
    rel_unc_cols = [c for c in df.columns if c.startswith('rel_unc_')]
    
    for i, (mean_col, sem_col, rel_col) in enumerate(zip(mean_cols, sem_cols, rel_unc_cols)):
        time_label = mean_col.replace('mean_', '')
        top_iso = df.nlargest(5, mean_col)
        
        print(f"\n  {time_label}:")
        print(f"  {'Isotope':<12} {'Activity (Bq/cm³)':<18} {'± SEM':<15} {'Rel. Unc.':<10}")
        print(f"  {'-'*55}")
        
        for _, row in top_iso.iterrows():
            iso = row['isotope']
            mean_val = row[mean_col]
            sem_val = row[sem_col] if sem_col in row else 0
            rel_val = row[rel_col] if rel_col in row else 0
            
            if mean_val > 1e-30:
                print(f"  {iso:<12} {mean_val:<18.4e} {sem_val:<15.4e} {rel_val*100:<10.1f}%")

## Step 8: Decay Heat Analysis

Analyze decay heat from the activation products.

In [13]:
# Load decay heat data
heat_data = {}

for csv_file in sorted(glob.glob(os.path.join(OUTPUT_DIR, 'tally_*_W_per_cm3.csv'))):
    tally_name = os.path.basename(csv_file).replace('_W_per_cm3.csv', '')
    material = material_names.get(tally_name, 'unknown')
    df = pd.read_csv(csv_file)
    heat_data[tally_name] = {'material': material, 'data': df}
    
    print(f"\n{tally_name} ({material}): {len(df)} contributors to decay heat")
    
    mean_cols = [c for c in df.columns if c.startswith('mean_')]
    if mean_cols:
        top = df.nlargest(5, mean_cols[0])[['isotope'] + mean_cols[:2]]
        print(top.to_string(index=False))

## Step 9: Summary Report

In [14]:
print("="*80)
print("ACTIVATION ANALYSIS SUMMARY")
print("="*80)
print(f"\nMCNP Input: {MCNP_INPUT}")
print(f"HDF5 File: {H5_FILE}")
print(f"Number of Mesh Tallies Processed: {len(activity_data)}")
print(f"Irradiation Time: 2 hours")
print(f"Cooling Times: shutdown, 3d, 7d, 45d, 1y")

print("\n" + "-"*60)
print("MATERIAL SUMMARY")
print("-"*60)

for tally_name, data in activity_data.items():
    material = data['material']
    df = data['data']
    
    print(f"\n{material} ({tally_name}):")
    print(f"  - Activated isotopes: {len(df)}")
    
    mean_cols = [c for c in df.columns if c.startswith('mean_')]
    if mean_cols:
        shutdown_col = mean_cols[0]
        total_shutdown = df[shutdown_col].sum()
        print(f"  - Total activity at shutdown: {total_shutdown:.3e} Bq/cm³")
        
        top3 = df.nlargest(3, shutdown_col)['isotope'].tolist()
        print(f"  - Top contributors: {', '.join([i.upper() for i in top3])}")

print("\n" + "="*80)
print("Analysis complete!")
print("="*80)

## Step 9b: Parse ALARA Output with Processing Tools

Use the `alara_output_processing` module from `tools/` to parse raw ALARA `.out` files into a structured `ALARADFrame` for easy filtering, analysis, and visualization.

In [15]:
# Parse all ALARA output files using alara_output_processing tools
print("Loading ALARA outputs using alara_output_processing tools...")

alara_output_dir = OUTPUT_DIR if 'OUTPUT_DIR' in dir() else Path('alara_output')
out_files = list(alara_output_dir.glob("**/tally_*_multizone.out"))

print(f"Found {len(out_files)} ALARA output files:")
for f in sorted(out_files):
    print(f"  - {f.relative_to(alara_output_dir)}")

runs_dict = {}
for out_file in sorted(out_files):
    tally_name = out_file.parent.name
    material = material_names.get(tally_name, tally_name)
    runs_dict[material] = str(out_file)

if runs_dict:
    datalib = DataLibrary()
    adf = datalib.make_entries(runs_dict, time_unit='s')
    
    print(f"\n✓ Loaded {len(adf)} data rows from {len(runs_dict)} tally outputs")
    print(f"  Unique materials/runs: {adf['run_lbl'].unique().tolist()}")
    print(f"  Variables: {[k for k in adf.VARIABLE_ENUM.keys()]}")
    print(f"  Cooling times (s): {sorted(adf['time'].unique().tolist())}")
    
    print(f"\nSample data (first 10 rows):")
    display(adf[['run_lbl', 'nuclide', 'variable', 'time', 'value', 'var_unit']].head(10))
else:
    print("No ALARA output files found!")
    adf = None

## Output Files Location

All output files are in the `alara_output/` directory:

**Per-tally subdirectories (`tally_XXXXX/`):**
- `*.inp` - ALARA input file
- `*.out` - ALARA output file (full results)
- `*_results.json` - Parsed results in JSON format
- `*_combined.flx` - Flux file (all voxels)
- `materials.matlib` - Material library
- `custom_elelib.txt` - Custom element library with MCNP isotopes

**CSV summary files:**
- `tally_XXXXX_Bq_per_cm3.csv` - Specific activity (Bq/cm³)
- `tally_XXXXX_W_per_cm3.csv` - Decay heat (W/cm³)
- `tally_XXXXX_atoms_per_cm3.csv` - Number density (atoms/cm³)
- `tally_XXXXX_g_per_cm3.csv` - Mass density (g/cm³)